In [80]:
import torch

data = torch.load("data.test.pt", map_location='cpu')

query_ids = torch.load("query_ids.pt", map_location='cpu')
prot_idx = torch.load("prot_idx.pt", map_location='cpu')


# '''DO NOT MODIFY
def find_items_per_group_per_query(data, query_ids, which_query, prot_idx):
    judgments_per_query = find_items_per_query(data, query_ids, which_query)
    prot_idx_per_query = find_items_per_query(prot_idx, query_ids, which_query)
    protected_items_per_query = judgments_per_query[prot_idx_per_query.bool()]
    nonprotected_items_per_query = judgments_per_query[~prot_idx_per_query.bool()]
    return judgments_per_query, protected_items_per_query, nonprotected_items_per_query

def find_items_per_query(data, query_ids, which_query):
    # import code
    # code.interact(local={**locals(), **globals()})
    return data[query_ids == which_query]

def normalized_exposure(group_data, all_data):
    return (torch.sum(topp_prot(group_data, all_data) / torch.log(torch.tensor(2.0)))) / group_data.size(0)

def topp_prot(group_items, all_items):
    return torch.exp(group_items) / torch.sum(torch.exp(all_items))

def exposure_diff(data, query_ids, which_query, prot_idx):
    judgments_per_query, protected_items_per_query, nonprotected_items_per_query = \
        find_items_per_group_per_query(data, query_ids, which_query, prot_idx)
    
    # print("Before", judgments_per_query.sum(), torch.exp(judgments_per_query))
    # return judgments_per_query, protected_items_per_query, nonprotected_items_per_query
    print(protected_items_per_query, nonprotected_items_per_query)
    print(torch.sum(torch.exp(judgments_per_query)), torch.exp(protected_items_per_query), torch.exp(nonprotected_items_per_query))
    

    exposure_prot = normalized_exposure(protected_items_per_query, judgments_per_query)
    exposure_nprot = normalized_exposure(nonprotected_items_per_query, judgments_per_query)
    exposure_diff = torch.max(torch.tensor(0.0), (exposure_nprot - exposure_prot)) ** 2
    # import code
    # code.interact(local={**locals(), **globals()})
    return exposure_diff

# '''


def vectorized_exposure_diff(data, query_ids, which_query, prot_idx):
    # Get unique query ids
    unique_queries = torch.unique(query_ids)
    
    # Create a mask for each unique query
    # query_masks = query_ids.unsqueeze(1) == unique_queries.unsqueeze(0)
    query_masks = query_ids.unsqueeze(1) == torch.tensor(which_query).unsqueeze(0)
    query_masks = query_masks.to(data.device)
    
    # Compute judgments per query
    judgments_per_query = (data.unsqueeze(1) * query_masks)
    
    # print("BEFORE", judgments_per_query.sum())
    
    # Compute protected and non-protected masks
    prot_masks = prot_idx.unsqueeze(1).to(data.device) * query_masks
    nprot_masks = (~prot_idx.bool()).unsqueeze(1).to(data.device) * query_masks
    
    
    # Compute protected and non-protected items per query
    protected_items_per_query = (judgments_per_query * prot_masks)
    nonprotected_items_per_query = (judgments_per_query * nprot_masks)
    
    print("Before exp", nprot_masks)
    
    # import code
    # code.interact(local={**locals(), **globals()})
    # We have to re-apply the masks since the exp() operation transforms 0s into 1s
    judgments_per_query = torch.sum(torch.exp(judgments_per_query) * query_masks)
    protected_items_per_query = torch.exp(protected_items_per_query) * prot_masks
    nonprotected_items_per_query = torch.exp(nonprotected_items_per_query) * nprot_masks
    # return judgments_per_query, protected_items_per_query, nonprotected_items_per_query
    
    print("AFTER", judgments_per_query.type())
    print("Non zeros -----")
    # print(judgments_per_query, protected_items_per_query.squeeze()[(judgments_per_query * prot_masks).squeeze().nonzero()], nonprotected_items_per_query.squeeze()[(judgments_per_query * nprot_masks).squeeze().nonzero()])
    # print(judgments_per_query.shape, protected_items_per_query.shape, nonprotected_items_per_query.shape)
    
    
    # Compute group sizes
    prot_group_sizes = prot_masks.sum(dim=0)
    nprot_group_sizes = nprot_masks.sum(dim=0)
    
    # Compute normalized exposure
    def modified_normalized_exposure(group_data, all_data, group_sizes):
        topp = group_data / all_data
        # import code
        # code.interact(local={**locals(), **globals()})
        return (topp / torch.log(torch.tensor(2.0))).sum(dim=0) / group_sizes
    
    exposure_prot = modified_normalized_exposure(protected_items_per_query, judgments_per_query, prot_group_sizes)
    exposure_nprot = modified_normalized_exposure(nonprotected_items_per_query, judgments_per_query, nprot_group_sizes)
    
    # Compute exposure difference
    exposure_diff = torch.max(torch.tensor(0.0), (exposure_nprot - exposure_prot)) ** 2
    
    return exposure_diff


In [81]:
# vect_judgments_per_query, vect_protected_items_per_query, vect_nonprotected_items_per_query = vectorized_exposure_diff(data, query_ids, prot_idx)
[vectorized_exposure_diff(data, query_ids, i, prot_idx) for i in range(100)]

Before exp tensor([[ True],
        [False],
        [ True],
        ...,
        [False],
        [False],
        [False]])
AFTER torch.FloatTensor
Non zeros -----
Before exp tensor([[False],
        [False],
        [False],
        ...,
        [False],
        [False],
        [False]])
AFTER torch.FloatTensor
Non zeros -----
Before exp tensor([[False],
        [False],
        [False],
        ...,
        [False],
        [False],
        [False]])
AFTER torch.FloatTensor
Non zeros -----
Before exp tensor([[False],
        [False],
        [False],
        ...,
        [False],
        [False],
        [False]])
AFTER torch.FloatTensor
Non zeros -----
Before exp tensor([[False],
        [False],
        [False],
        ...,
        [False],
        [False],
        [False]])
AFTER torch.FloatTensor
Non zeros -----
Before exp tensor([[False],
        [False],
        [False],
        ...,
        [False],
        [False],
        [False]])
AFTER torch.FloatTensor
Non zeros ----

[tensor([3.4035e-10], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([1.5457e-07], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([2.6512e-07], grad_fn=<PowBackward0>),
 tensor([3.2502e-07], grad_fn=<PowBackward0>),
 tensor([1.3025e-09], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([1.4785e-07], grad_fn=<PowBackward0>),
 tensor([1.8035e-07], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([2.9027e-09], grad_fn=<PowBackward0>),
 tensor([1.1154e-11], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([1.0683e-10], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([4.8087e-08], grad_fn=<PowBackward0>),
 tensor([4.0905e-08], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor([0.], grad_fn=<PowBackward0>),
 tensor

In [79]:
# judgments_per_query, protected_items_per_query, nonprotected_items_per_query = exposure_diff(data, query_ids, torch.tensor(0), prot_idx)
[exposure_diff(data, query_ids, torch.tensor(i), prot_idx) for i in range(100)]

tensor([6.0674e-01, 7.8310e-01, 2.3847e-02, 8.3762e-01, 9.2715e-01, 1.4751e-04,
        8.7321e-03, 7.9352e-01, 8.2325e-02, 5.8353e-02, 7.8492e-01, 6.2241e-01,
        9.7203e-01, 4.8828e-01, 7.7558e-01, 2.4933e-01, 2.8219e-01, 5.2563e-01,
        3.9702e-01, 5.2490e-01, 6.8372e-01, 3.4363e-01, 8.5211e-01, 3.1674e-01,
        8.9803e-01, 3.5085e-01, 4.8255e-01, 2.2618e-01, 4.2474e-01, 4.9062e-01,
        4.8766e-01, 4.9702e-01, 4.9233e-01, 4.9232e-01, 5.0930e-01, 4.8021e-01,
        5.0445e-01, 5.0809e-01, 4.9941e-01, 5.0219e-01, 4.9629e-01, 4.9478e-01,
        5.1147e-01, 5.0111e-01, 4.8090e-01, 4.9884e-01, 4.9968e-01, 5.1780e-01,
        4.9218e-01, 4.9217e-01, 5.0274e-01], grad_fn=<IndexBackward0>) tensor([1.1880e-06, 9.9998e-01, 7.3481e-03, 9.6071e-01, 9.4419e-01, 4.1545e-01,
        1.0000e+00, 2.1111e-01, 1.7159e-01, 9.0816e-01, 2.0986e-01, 1.2617e-01,
        9.7724e-01, 8.2419e-01, 4.7058e-05, 9.9695e-01, 9.8703e-01, 8.7322e-01,
        5.2801e-01, 1.1832e-03, 1.9581e-01, 3.928

[tensor(3.4038e-10, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(1.5457e-07, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(2.6513e-07, grad_fn=<PowBackward0>),
 tensor(3.2502e-07, grad_fn=<PowBackward0>),
 tensor(1.3025e-09, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(1.4785e-07, grad_fn=<PowBackward0>),
 tensor(1.8035e-07, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(2.9027e-09, grad_fn=<PowBackward0>),
 tensor(1.1154e-11, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(1.0683e-10, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(4.8087e-08, grad_fn=<PowBackward0>),
 tensor(4.0906e-08, grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(0., grad_fn=<PowBackward0>),
 tensor(0., grad

In [66]:
vect_nonprotected_items_per_query

tensor([[1.],
        [0.],
        [1.],
        ...,
        [0.],
        [0.],
        [0.]], grad_fn=<MulBackward0>)

In [29]:
judgments_per_query

tensor([1.1880e-06, 6.0674e-01, 9.9998e-01, 7.3481e-03, 9.6071e-01, 9.4419e-01,
        7.8310e-01, 4.1545e-01, 2.3847e-02, 1.0000e+00, 2.1111e-01, 1.7159e-01,
        8.3762e-01, 9.2715e-01, 1.4751e-04, 9.0816e-01, 2.0986e-01, 1.2617e-01,
        9.7724e-01, 8.2419e-01, 4.7058e-05, 9.9695e-01, 8.7321e-03, 9.8703e-01,
        8.7322e-01, 7.9352e-01, 8.2325e-02, 5.8353e-02, 5.2801e-01, 1.1832e-03,
        1.9581e-01, 3.9283e-01, 4.0790e-01, 9.7177e-01, 7.8492e-01, 1.8941e-01,
        5.6569e-01, 8.9213e-01, 2.5991e-01, 3.8136e-01, 7.5398e-01, 1.8256e-01,
        6.2241e-01, 9.3918e-01, 9.7203e-01, 4.8828e-01, 7.7558e-01, 8.8519e-03,
        8.1608e-01, 3.9106e-01, 1.7667e-01, 6.5379e-01, 4.5066e-02, 2.3221e-01,
        2.4933e-01, 3.0799e-01, 2.8219e-01, 6.2848e-01, 5.2563e-01, 6.8764e-01,
        3.9702e-01, 5.4584e-01, 5.2023e-01, 4.9328e-01, 5.2490e-01, 6.1249e-02,
        4.0491e-01, 6.8372e-01, 3.4363e-01, 8.5211e-01, 3.1674e-01, 7.0610e-01,
        8.9803e-01, 3.5085e-01, 4.3382e-

In [63]:
vect_nonprotected_items_per_query.squeeze()[vect_nonprotected_items_per_query.squeeze().nonzero()].squeeze()

tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
        1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
       grad_fn=<SqueezeBackward0>)

In [60]:
torch.exp(protected_items_per_query)

tensor([1.8344, 2.1883, 1.0241, 2.3109, 2.5273, 1.0001, 1.0088, 2.2112, 1.0858,
        1.0601, 2.1922, 1.8634, 2.6433, 1.6295, 2.1719, 1.2832, 1.3260, 1.6915,
        1.4874, 1.6903, 1.9812, 1.4101, 2.3446, 1.3727, 2.4548, 1.4203, 1.6202,
        1.2538, 1.5292, 1.6333, 1.6285, 1.6438, 1.6361, 1.6361, 1.6641, 1.6164,
        1.6561, 1.6621, 1.6478, 1.6523, 1.6426, 1.6401, 1.6677, 1.6506, 1.6175,
        1.6468, 1.6482, 1.6783, 1.6359, 1.6359, 1.6532],
       grad_fn=<ExpBackward0>)

In [31]:
nonprotected_items_per_query

tensor([1.1880e-06, 9.9998e-01, 7.3481e-03, 9.6071e-01, 9.4419e-01, 4.1545e-01,
        1.0000e+00, 2.1111e-01, 1.7159e-01, 9.0816e-01, 2.0986e-01, 1.2617e-01,
        9.7724e-01, 8.2419e-01, 4.7058e-05, 9.9695e-01, 9.8703e-01, 8.7322e-01,
        5.2801e-01, 1.1832e-03, 1.9581e-01, 3.9283e-01, 4.0790e-01, 9.7177e-01,
        1.8941e-01, 5.6569e-01, 8.9213e-01, 2.5991e-01, 3.8136e-01, 7.5398e-01,
        1.8256e-01, 9.3918e-01, 8.8519e-03, 8.1608e-01, 3.9106e-01, 1.7667e-01,
        6.5379e-01, 4.5066e-02, 2.3221e-01, 3.0799e-01, 6.2848e-01, 6.8764e-01,
        5.4584e-01, 5.2023e-01, 4.9328e-01, 6.1249e-02, 4.0491e-01, 7.0610e-01,
        4.3382e-01, 2.6612e-01, 4.3143e-01, 3.2924e-01, 4.1261e-01, 5.2949e-01,
        4.0942e-01, 5.1492e-01, 4.1706e-01, 4.6328e-01, 4.6969e-01, 4.9646e-01,
        5.0898e-01, 4.9956e-01, 4.9255e-01, 5.0025e-01, 4.9339e-01, 4.9854e-01,
        5.0710e-01, 4.9238e-01, 4.9560e-01, 9.9374e-01, 5.0488e-01, 4.9660e-01,
        5.0042e-01, 5.6615e-01, 4.9431e-